In [3]:
import torch
import torch.nn as nn

class PostionEncode(nn.Module):
    def __init__(self, d_model, max_len=2048):
        super().__init__()
        self.d_model = d_model
        self.max_len = max_len
        position = torch.arange(0,max_len).unsqueeze(1)

        w = 1 / (10000 ** (torch.arange(0,d_model,2)/d_model))

        pe = torch.zeros(max_len, d_model)

        pe[:,0::2] = torch.sin(position*w)
        pe[:,1::2] = torch.cos(position*w)

        self.register_buffer("pe",pe)

    def forward(self,x):
        s = x.size(1)

        return x + self.pe[:s]

x = torch.randn(32, 50, 512) 
pos_en = PostionEncode(512)

print(pos_en(x).shape)


torch.Size([32, 50, 512])


In [11]:
import torch
import torch.nn as nn
import torch.nn.functional as F


def rotate_half(x):
    x1,x2 = x.chunk(2,dim=-1)

    return torch.cat((-x2,x1),dim=-1)

def apply_rope(q,k,sin,cos):

    q_embed = q * cos + rotate_half(q)*sin
    k_embed = k * cos + rotate_half(k)*sin

    return q_embed, k_embed

class Rope(nn.Module):
    def __init__(self,d_model, max_len=2048):
        super().__init__()
        self.d_model = d_model
        self.max_len = max_len

        pos = torch.arange(0,max_len).float().unsqueeze(1)

        inv_freq = 1 / (10000 ** (torch.arange(0,d_model,2).float()/d_model))

        freqs = pos @ inv_freq.unsqueeze(0)

        freqs = torch.cat((freqs,freqs), dim=-1)
        # [max_len, d_model]
        self.register_buffer("sin_cached", freqs.sin())
        self.register_buffer("cos_cached", freqs.cos())

    def forward(self, q, k):
        # q [b,num_head,s,head_dim]
        s = q.shape[2]
        sin = self.sin_cached[:s,:].unsqueeze(0).unsqueeze(0)
        cos = self.cos_cached[:s,:].unsqueeze(0).unsqueeze(0)

        return apply_rope(q,k,sin,cos)

q = torch.rand(4,4,128,128)
k = torch.rand(4,4,128,128)
rope = Rope(128)

q_embd, k_embd = rope(q,k)
print(q_embd.shape)

torch.Size([4, 4, 128, 128])
